# 测试 Embedding 和 Reranker 服务

- Embedding 服务: http://localhost:8200/v1
- Reranker 服务: http://localhost:8201/v1

In [1]:
import sys
sys.path.insert(0, '/home/jiazixiao.jzx/MedAgent')

from src.medagent import EmbeddingClient, RerankerClient

## 1. 测试 Embedding 服务

In [ ]:
# 创建 Embedding 客户端
emb_client = EmbeddingClient(
    base_url="http://localhost:8200/v1",
    model_name="Qwen3-embedding"
)
print("Embedding 客户端已创建")

In [ ]:
# 测试单条 embedding
text = "高血压是一种常见的慢性疾病"
emb = emb_client.embed(text)
print(f"文本: {text}")
print(f"向量维度: {len(emb)}")
print(f"前 10 维: {emb[:10]}")

In [ ]:
# 测试批量 embedding
texts = [
    "高血压的治疗方案",
    "糖尿病的饮食控制",
    "感冒的症状和治疗"
]
embs = emb_client.embed_batch(texts)
print(f"批量处理 {len(texts)} 条文本")
print(f"每条向量维度: {len(embs[0])}")

In [ ]:
# 测试相似度计算
score = EmbeddingClient.cosine_similarity(embs[0], embs[1])
print(f"'高血压' 与 '糖尿病' 相似度: {score:.4f}")

score2 = EmbeddingClient.cosine_similarity(embs[0], embs[0])
print(f"'高血压' 与自身相似度: {score2:.4f}")

In [ ]:
# 测试检索功能
query = "血压高怎么办"
docs = [
    "高血压患者应该低盐饮食，适当运动",
    "感冒需要多休息，多喝水",
    "糖尿病要控制血糖，定期检查"
]

results = emb_client.search(query, docs, top_k=3)
print(f"查询: {query}")
print(f"\n检索结果:")
for idx, score, doc in results:
    print(f"[{idx}] {score:.4f}: {doc}")

## 2. 测试 Reranker 服务

Reranker 使用 Qwen3-Reranker 模型，通过以下方式计算相关性：
1. 构建 prompt 让模型输出 yes/no
2. 使用 `allowed_token_ids` 限制输出
3. 从 logprobs 计算 `P(yes) / (P(yes) + P(no))` 作为分数

In [2]:
# 创建 Reranker 客户端
rerank_client = RerankerClient(
    base_url="http://localhost:8201/v1",
    model_name="Qwen3-reranker"
)
print("Reranker 客户端已创建")

[Reranker] Warning: Could not get token ids, using defaults: SyncAPIClient.post() got an unexpected keyword argument 'json'
Reranker 客户端已创建


In [3]:
# 查看 token id 配置
print(f"true_token_id (yes): {rerank_client._true_token_id}")
print(f"false_token_id (no): {rerank_client._false_token_id}")

true_token_id (yes): 9453
false_token_id (no): 2732


In [4]:
# 测试单条打分
query = "高血压怎么治疗?"
doc = "高血压患者需要长期服用降压药物，同时注意低盐饮食和规律运动。"

score = rerank_client.compute_score(query, doc)
print(f"查询: {query}")
print(f"文档: {doc}")
print(f"相关性分数: {score:.4f}")

查询: 高血压怎么治疗?
文档: 高血压患者需要长期服用降压药物，同时注意低盐饮食和规律运动。
相关性分数: 0.5000


In [5]:
# 测试不同相关性的文档
query = "高血压怎么治疗?"
docs = [
    "高血压患者需要长期服用降压药物，同时注意低盐饮食和规律运动。",
    "感冒是由病毒引起的呼吸道感染，症状包括发热、咳嗽、流鼻涕。",
    "糖尿病是一种代谢性疾病，主要表现为血糖升高。",
    "血压是指血液在血管内流动时对血管壁产生的压力。"
]

print(f"查询: {query}")
print("\n各文档相关性分数:")
for i, doc in enumerate(docs):
    score = rerank_client.compute_score(query, doc)
    print(f"[{i+1}] {score:.4f}: {doc[:50]}...")

查询: 高血压怎么治疗?

各文档相关性分数:
[1] 0.5000: 高血压患者需要长期服用降压药物，同时注意低盐饮食和规律运动。...
[2] 0.5000: 感冒是由病毒引起的呼吸道感染，症状包括发热、咳嗽、流鼻涕。...
[3] 0.5000: 糖尿病是一种代谢性疾病，主要表现为血糖升高。...
[4] 0.5000: 血压是指血液在血管内流动时对血管壁产生的压力。...


In [6]:
# 测试批量打分
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

scores = rerank_client.compute_score_batch(queries, documents)
for i, (q, d, s) in enumerate(zip(queries, documents, scores)):
    print(f"[{i+1}] Score: {s:.4f}")
    print(f"    Query: {q}")
    print(f"    Doc: {d[:50]}...")

[1] Score: 0.5000
    Query: What is the capital of China?
    Doc: The capital of China is Beijing....
[2] Score: 0.5000
    Query: Explain gravity
    Doc: Gravity is a force that attracts two bodies toward...


In [7]:
# 测试批量重排序
query = "糖尿病的并发症有哪些?"
docs = [
    "糖尿病的常见并发症包括视网膜病变、肾病、神经病变等。",
    "高血压会导致心脏病、中风等严重后果。",
    "糖尿病足是糖尿病患者常见的并发症之一。",
    "感冒通常一周左右可以自愈。"
]

results = rerank_client.rerank(query, docs, top_k=3)
print(f"查询: {query}")
print(f"\n重排序结果 (Top 3):")
for idx, score, doc in results:
    print(f"[{idx+1}] {score:.4f}: {doc}")

查询: 糖尿病的并发症有哪些?

重排序结果 (Top 3):
[1] 0.5000: 糖尿病的常见并发症包括视网膜病变、肾病、神经病变等。
[2] 0.5000: 高血压会导致心脏病、中风等严重后果。
[3] 0.5000: 糖尿病足是糖尿病患者常见的并发症之一。


## 3. 测试知识库 (如果已构建)

In [ ]:
import os

db_path = "/home/jiazixiao.jzx/MedAgent/data/knowledge_db"
if os.path.exists(db_path):
    from src.medagent import KnowledgeBase
    
    kb = KnowledgeBase(
        embedding_client=emb_client,
        reranker_client=rerank_client
    )
    kb.load(db_path)
    
    # 测试检索
    query = "高血压的治疗方案"
    results = kb.search(query, top_k=5, rerank_top_n=3)
    
    print(f"查询: {query}")
    print(f"\n找到 {len(results)} 条结果:")
    for i, (doc, score, meta) in enumerate(results):
        print(f"\n[{i+1}] 相似度: {score:.4f}")
        print(f"{doc[:300]}...")
else:
    print(f"知识库目录不存在: {db_path}")

In [ ]:
# 测试知识库检索 + 摘要
if 'kb' in dir() and kb.count() > 0:
    query = "急性心肌梗死的诊断标准"
    summary = kb.search_with_summary(query, top_k=10, rerank_top_n=3, max_length=500)
    print(f"查询: {query}")
    print(f"\n知识摘要:")
    print(summary)
else:
    print("知识库为空或未加载")